#  MedTrack_DV - Hospital Operations & Patient Analytics Dashboard

## Milestone 1: Data Collection & Integration 

## Module 1: Hospital Data Collection
### Objective
The objective of this milestone is to collect, integrate, and prepare hospital operational datasets for healthcare analytics.

### Datasets Used
- Patient Admission Dataset
- Hospital Information Dataset
- Department Resource Dataset

### Expected Deliverable
A unified master dataset (`hospital_raw_data.csv`) for further data cleaning and KPI engineering.

# Step 1: Import Required Libraries
Import the necessary Python libraries for data manipulation and analysis.

In [30]:
import pandas as pd
import numpy as np

# Step 2: Load the Datasets

Load the three datasets into Pandas DataFrames.

- Patient Admissions
- Hospital Information
- Department Resources

In [2]:
patient = pd.read_excel(r"D:\INFOSYS_PROJECT\archive\proper dataset\patient_admissions.xlsx")

hospital = pd.read_excel(r"D:\INFOSYS_PROJECT\archive\proper dataset\hospital_info.xlsx")

department = pd.read_excel(r"D:\INFOSYS_PROJECT\archive\proper dataset\department_resources.xlsx")

# Step 3: Verify Dataset Dimensions

Check the number of rows and columns in each dataset to ensure they have been loaded successfully.

In [3]:
print(patient.shape)
print(hospital.shape)
print(department.shape)

(15757, 58)
(15, 7)
(105, 10)


# Step 4: Explore Dataset Columns

Display the column names of all datasets to understand their structure before integration.

In [4]:
patient.columns

Index(['SNO', 'MRD No.', 'Hospital_ID', 'Department_ID', 'D.O.A', 'D.O.D',
       'AGE', 'GENDER', 'RURAL', 'TYPE OF ADMISSION-EMERGENCY/OPD',
       'month year', 'DURATION OF STAY', 'duration of intensive unit stay',
       'OUTCOME', 'SMOKING ', 'ALCOHOL', 'DM', 'HTN', 'CAD', 'PRIOR CMP',
       'CKD', 'HB', 'TLC', 'PLATELETS', 'GLUCOSE', 'UREA', 'CREATININE', 'BNP',
       'RAISED CARDIAC ENZYMES', 'EF', 'SEVERE ANAEMIA', 'ANAEMIA',
       'STABLE ANGINA', 'ACS', 'STEMI', 'ATYPICAL CHEST PAIN', 'HEART FAILURE',
       'HFREF', 'HFNEF', 'VALVULAR', 'CHB', 'SSS', 'AKI', 'CVA INFRACT',
       'CVA BLEED', 'AF', 'VT', 'PSVT', 'CONGENITAL', 'UTI',
       'NEURO CARDIOGENIC SYNCOPE', 'ORTHOSTATIC', 'INFECTIVE ENDOCARDITIS',
       'DVT', 'CARDIOGENIC SHOCK', 'SHOCK', 'PULMONARY EMBOLISM',
       'CHEST INFECTION'],
      dtype='object')

In [5]:
hospital.columns

Index(['Hospital_ID', 'Hospital_Name', 'Hospital_Type', 'City', 'State',
       'Total_Beds', 'ICU_Beds'],
      dtype='object')

In [6]:
department.columns

Index(['Department_ID', 'Hospital_ID', 'Department', 'Doctor_ID',
       'Doctor_Name', 'Nurses', 'Staff_Count', 'Beds_Available',
       'Beds_Occupied', 'Equipment'],
      dtype='object')

# Step 5: Remove Existing Hospital and Department IDs

Remove the existing Hospital_ID and Department_ID columns from the patient dataset.

These values will be regenerated to simulate a realistic multi-hospital environment.

In [7]:
patient = patient.drop(
    columns=["Hospital_ID", "Department_ID"],
    errors="ignore"
)

In [8]:
patient.columns

Index(['SNO', 'MRD No.', 'D.O.A', 'D.O.D', 'AGE', 'GENDER', 'RURAL',
       'TYPE OF ADMISSION-EMERGENCY/OPD', 'month year', 'DURATION OF STAY',
       'duration of intensive unit stay', 'OUTCOME', 'SMOKING ', 'ALCOHOL',
       'DM', 'HTN', 'CAD', 'PRIOR CMP', 'CKD', 'HB', 'TLC', 'PLATELETS',
       'GLUCOSE', 'UREA', 'CREATININE', 'BNP', 'RAISED CARDIAC ENZYMES', 'EF',
       'SEVERE ANAEMIA', 'ANAEMIA', 'STABLE ANGINA', 'ACS', 'STEMI',
       'ATYPICAL CHEST PAIN', 'HEART FAILURE', 'HFREF', 'HFNEF', 'VALVULAR',
       'CHB', 'SSS', 'AKI', 'CVA INFRACT', 'CVA BLEED', 'AF', 'VT', 'PSVT',
       'CONGENITAL', 'UTI', 'NEURO CARDIOGENIC SYNCOPE', 'ORTHOSTATIC',
       'INFECTIVE ENDOCARDITIS', 'DVT', 'CARDIOGENIC SHOCK', 'SHOCK',
       'PULMONARY EMBOLISM', 'CHEST INFECTION'],
      dtype='object')

# Step 6: Assign Hospital IDs

Randomly assign each patient to one of the 15 hospitals using weighted probabilities.

This creates a realistic distribution of patient admissions across hospitals.

In [9]:
import numpy as np

hospital_ids = hospital["Hospital_ID"].tolist()

probabilities = [
    0.10, 0.08, 0.12, 0.08, 0.05,
    0.05, 0.07, 0.06, 0.05, 0.09,
    0.07, 0.06, 0.06, 0.03, 0.03
]

patient["Hospital_ID"] = np.random.choice(
    hospital_ids,
    size=len(patient),
    p=probabilities
)

# Step 7: Verify Hospital Distribution

Display the number of patients assigned to each hospital.

In [10]:
patient["Hospital_ID"].value_counts()

Hospital_ID
H003    1867
H001    1614
H010    1370
H002    1279
H004    1253
H011    1093
H007    1065
H008     990
H013     974
H012     932
H005     886
H009     783
H006     731
H014     482
H015     438
Name: count, dtype: int64

# Step 8: Analyze Clinical Columns

Inspect the clinical condition columns used for assigning departments.

In [11]:
patient[["CAD","CKD","ACS","STEMI"]].head(10)

,CAD,CKD,ACS,STEMI
0,0,0,1,0
1,1,0,0,0
2,1,0,0,0
3,1,0,0,0
4,0,0,0,0
5,1,0,1,0
6,1,0,1,1
7,0,0,0,0
8,0,0,1,0
9,1,0,0,0


In [12]:
print(patient["CAD"].value_counts())

CAD
1    10551
0     5206
Name: count, dtype: int64


# Step 9: Assign Departments

Assign each patient to an appropriate hospital department based on their medical condition.

In [13]:
def assign_department(row):

    # Cardiology
    if (
        row["CAD"] == 1 or
        row["ACS"] == 1 or
        row["STEMI"] == 1 or
        row["HEART FAILURE"] == 1 or
        row["HFREF"] == 1 or
        row["HFNEF"] == 1 or
        row["STABLE ANGINA"] == 1
    ):
        return "Cardiology"

    # Neurology
    elif (
        row["CVA INFRACT"] == 1 or
        row["CVA BLEED"] == 1
    ):
        return "Neurology"

    # Nephrology
    elif (
        row["CKD"] == 1 or
        row["AKI"] == 1
    ):
        return "Nephrology"

    # ICU
    elif (
        row["CHB"] == 1 or
        row["SSS"] == 1 or
        row["CARDIOGENIC SHOCK"] == 1 or
        row["SHOCK"] == 1
    ):
        return "ICU"

    # Emergency
    elif (
        row["PULMONARY EMBOLISM"] == 1
    ):
        return "Emergency"

    # Default
    else:
        return "General Medicine"

# Step 10: Apply Department Assignment

In [14]:
patient["Department"] = patient.apply(assign_department, axis=1)

# Step 11: Verify Department Distribution

Display the number of patients assigned to each department.

In [15]:
patient["Department"].value_counts()

Department
Cardiology          12971
General Medicine     1845
Nephrology            457
ICU                   215
Neurology             155
Emergency             114
Name: count, dtype: int64

# Step 12: Assign Department IDs

Map the generated departments to their corresponding Department IDs.

In [16]:
dept_lookup = department[
    ["Hospital_ID", "Department", "Department_ID"]
]

In [17]:
patient = patient.merge(
    dept_lookup,
    on=["Hospital_ID", "Department"],
    how="left"
)

In [18]:
patient.head()

,SNO,MRD No.,D.O.A,D.O.D,AGE,GENDER,RURAL,TYPE OF ADMISSION-EMERGENCY/OPD,month year,DURATION OF STAY,...,ORTHOSTATIC,INFECTIVE ENDOCARDITIS,DVT,CARDIOGENIC SHOCK,SHOCK,PULMONARY EMBOLISM,CHEST INFECTION,Hospital_ID,Department,Department_ID
0,1,234735,2017-01-04 00:00:00,2017-03-04 00:00:00,81,M,R,E,2017-04-01,3,...,0,0,0,0,0,0,0,H003,Cardiology,D015
1,2,234696,2017-01-04 00:00:00,2017-05-04 00:00:00,65,M,R,E,2017-04-01,5,...,0,0,0,0,0,0,0,H002,Cardiology,D008
2,3,234882,2017-01-04 00:00:00,2017-03-04 00:00:00,53,M,U,E,2017-04-01,3,...,0,0,0,0,0,0,0,H015,Cardiology,D099
3,4,234635,2017-01-04 00:00:00,2017-08-04 00:00:00,67,F,U,E,2017-04-01,8,...,0,0,0,0,0,0,0,H006,Cardiology,D036
4,5,234486,2017-01-04 00:00:00,4/23/2017,60,F,U,E,2017-04-01,23,...,0,0,0,0,0,0,0,H011,General Medicine,D075


# Step 13: Validate Department ID Assignment

Ensure that every patient has been assigned a valid Department ID.

In [19]:
print(patient["Department_ID"].isna().sum())

0


In [20]:
patient["Department_ID"].isna().sum()

np.int64(0)

# Step 14: Merge Hospital Information

Combine the patient dataset with hospital information using Hospital_ID.

In [21]:
master = patient.merge(
    hospital,
    on="Hospital_ID",
    how="left"
)

In [22]:
master.shape

(15757, 65)

# Step 15: Merge Department Resource Information

Merge doctor details, staff information, bed availability, and equipment data.

In [23]:
dept_details = department[
    [
        "Hospital_ID",
        "Department_ID",
        "Doctor_ID",
        "Doctor_Name",
        "Nurses",
        "Staff_Count",
        "Beds_Available",
        "Beds_Occupied",
        "Equipment"
    ]
]

In [24]:
master = master.merge(
    dept_details,
    on=["Hospital_ID","Department_ID"],
    how="left"
)

# Step 16: Verify the Master Dataset

Display the first few rows of the integrated dataset.

In [25]:
master.head()

,SNO,MRD No.,D.O.A,D.O.D,AGE,GENDER,RURAL,TYPE OF ADMISSION-EMERGENCY/OPD,month year,DURATION OF STAY,...,State,Total_Beds,ICU_Beds,Doctor_ID,Doctor_Name,Nurses,Staff_Count,Beds_Available,Beds_Occupied,Equipment
0,1,234735,2017-01-04 00:00:00,2017-03-04 00:00:00,81,M,R,E,2017-04-01,3,...,Delhi,1000,200,DOC015,Dr. Suresh,27,62,80,68,ECG Machine
1,2,234696,2017-01-04 00:00:00,2017-05-04 00:00:00,65,M,R,E,2017-04-01,5,...,Karnataka,450,70,DOC008,Dr. Rahul,26,61,80,69,ECG Machine
2,3,234882,2017-01-04 00:00:00,2017-03-04 00:00:00,53,M,U,E,2017-04-01,3,...,Haryana,1250,250,DOC099,Dr. Anita,27,64,80,68,ECG Machine
3,4,234635,2017-01-04 00:00:00,2017-08-04 00:00:00,67,F,U,E,2017-04-01,8,...,Tamil Nadu,400,60,DOC036,Dr. Arjun,26,60,80,65,ECG Machine
4,5,234486,2017-01-04 00:00:00,4/23/2017,60,F,U,E,2017-04-01,23,...,Karnataka,700,100,DOC075,Dr. Suresh,26,60,100,86,Patient Monitor


# Step 17: Check Missing Values

Verify that the merged dataset does not contain missing values in important operational fields.

In [26]:
master.isnull().sum()

SNO               0
MRD No.           0
D.O.A             0
D.O.D             0
AGE               0
                 ..
Nurses            0
Staff_Count       0
Beds_Available    0
Beds_Occupied     0
Equipment         0
Length: 72, dtype: int64

# Step 18: Save the Integrated Dataset

Export the integrated hospital dataset for the next milestone.

**Output File:**
- hospital_raw_data.csv

In [27]:
master.to_csv(
    "hospital_raw_data.csv",
    index=False
)

# ✅ Milestone 1 (Module 1: Hospital Data Collection) Completed

## Achievements

- Successfully collected three healthcare datasets.
- Integrated patient, hospital, and department datasets.
- Assigned Hospital IDs and Department IDs.
- Created a unified healthcare master dataset.
- Exported the integrated dataset for further processing.

### Deliverable

**hospital_raw_data.csv**